
# 04. Traditional Machine Learning Training

## Objective

Train and optimize traditional machine learning models for peptide MIC prediction using previously extracted sequence features.

## Models

- Random Forest (RF)
- Support Vector Regression (SVR)
- Decision Tree (DT)
- K-Nearest Neighbors (KNN)

## Experimental Design

- Datasets: EC and SA
- Features: AAC, DPC, CTDC, CTDT, CTDD, QSOrder
- Random seeds: 42, 43, 44, 45, 46
- Hyperparameter selection: Mean validation MSE
- Regression target: pMIC

## Workflow

1. Load extracted features and predefined validation splits.
2. Initialize ML models and hyperparameter grids.
3. Perform hyperparameter optimization across five random seeds.
4. Select the best hyperparameters using mean validation MSE.
5. Retrain the selected models on the full training dataset.
6. Evaluate the independent test dataset.
7. Save models, hyperparameters, predictions, and evaluation metrics.

## Reproducibility

All experiments use predefined data splits, fixed random seeds, and consistent evaluation metrics.


In [1]:

import importlib

import configs.ml as ml_config
from src.ml import create_model
from configs.common import RANDOM_STATE

# Reload configuration
importlib.reload(ml_config)

# Initialize models
for name in ml_config.ML_MODELS:

    params = {
        key: values[0]
        for key, values in ml_config.PARAM_GRIDS[name].items()
    }

    model = create_model(name, params, RANDOM_STATE)

    print(f"\n[{name}]")
    print(model)



[RF]
RandomForestRegressor(n_jobs=-1, random_state=42)

[SVR]
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', SVR(C=0.1, epsilon=0.01))])

[DT]
DecisionTreeRegressor(random_state=42)

[KNN]
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsRegressor(n_neighbors=3, p=1))])


In [2]:
import importlib
import src.ml as ml

# Reload updated module
importlib.reload(ml)

# Load EC AAC dataset
X_train, y_train, X_test, y_test = ml.load_ml_data(
    target="EC",
    feature="AAC"
)

# Load seed 42 validation split
train_idx, valid_idx = ml.load_validation_split(
    target="EC",
    seed=42
)

# Display dataset dimensions
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")

print(f"\nTraining samples  : {len(train_idx):,}")
print(f"Validation samples: {len(valid_idx):,}")

# Verify split
assert len(train_idx) + len(valid_idx) == len(X_train)
assert len(set(train_idx) & set(valid_idx)) == 0

print("\n[OK] ML data and validation split loaded.")


X_train: (7583, 20)
y_train: (7583,)
X_test : (843, 20)
y_test : (843,)

Training samples  : 6,824
Validation samples: 759

[OK] ML data and validation split loaded.


In [3]:
import importlib
import src.ml as ml

importlib.reload(ml)

# Small test grid
test_grid = {
    "n_estimators": [100, 300],
    "max_depth": [10],
    "min_samples_leaf": [1]
}

# Run test
grid_results, best_result = ml.run_grid_search(
    target="EC",
    feature="AAC",
    model_name="RF",
    param_grid=test_grid
)

# Preview results
display(grid_results.head(10))


,Model,Feature,Seed,Parameters,Validation_MSE
0,RF,AAC,42,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.348000
1,RF,AAC,43,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.368277
2,RF,AAC,44,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.343649
3,RF,AAC,45,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.309148
4,RF,AAC,46,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.314757
5,RF,AAC,42,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.349514
6,RF,AAC,43,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.366167
7,RF,AAC,44,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.341398
8,RF,AAC,45,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.307590
9,RF,AAC,46,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""n_es...",0.313190


In [4]:
import importlib
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import ParameterGrid

import src.ml as ml
from configs.ml import ML_MODELS, FEATURES, PARAM_GRIDS
from configs.common import RESULT_DIR, RANDOM_SEEDS

importlib.reload(ml)

# Define all experiments
tasks = [
    (target, feature, model)
    for target in ["EC", "SA"]
    for feature in FEATURES
    for model in ML_MODELS
]

# Calculate total model fits
total_fits = sum(
    len(ParameterGrid(PARAM_GRIDS[model])) * len(RANDOM_SEEDS)
    for target, feature, model in tasks
)

all_best = []

# Run full ML optimization
with tqdm(
    total=total_fits,
    desc="ML Optimization",
    unit="fit",
    dynamic_ncols=False,
    ncols=120,
    mininterval=1
) as pbar:

    for target, feature, model in tasks:

        results, best = ml.run_grid_search(
            target=target,
            feature=feature,
            model_name=model,
            param_grid=PARAM_GRIDS[model],
            progress=pbar
        )

        # Summarize validation results
        summary = (
            results
            .groupby(["Model", "Feature", "Parameters"])
            ["Validation_MSE"]
            .agg(["mean", "std"])
            .reset_index()
            .rename(columns={
                "mean": "Mean_Validation_MSE",
                "std": "SD_Validation_MSE"
            })
            .sort_values("Mean_Validation_MSE")
        )

        output_dir = RESULT_DIR / "traditional_ml" / target
        output_dir.mkdir(parents=True, exist_ok=True)

        summary.to_csv(
            output_dir / f"{model}_{feature}_validation_summary.csv",
            index=False
        )

        # Record best hyperparameters
        all_best.append({
            "Dataset": target,
            "Feature": feature,
            "Model": model,
            "Best_Parameters": str(best["Best_Parameters"]),
            "Validation_MSE": best["Best_Validation_MSE"]
        })

        pd.DataFrame(all_best).to_csv(
            RESULT_DIR / "traditional_ml"
            / "best_hyperparameters_summary.csv",
            index=False
        )

        # Show best validation result
        pbar.set_postfix(
            Best_Val_MSE=f"{best['Best_Validation_MSE']:.6f}",
            refresh=True
        )

        tqdm.write(
            f"[{target} | {feature} | {model}] "
            f"Best Val MSE: {best['Best_Validation_MSE']:.6f} | "
            f"Best Params: {best['Best_Parameters']}"
        )

print("\n[OK] All ML hyperparameter optimization completed.")

EC | AAC | SVR:   2%|▍                       | 90/4920 [01:16<1:29:03,  1.11s/fit, combo=1/24, seed=42, status=training]

[EC | AAC | RF] Best Val MSE: 0.292002 | Best Params: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 500}


EC | AAC | DT:   4%|▉                      | 212/4920 [10:44<14:24:49, 11.02s/fit, combo=1/24, seed=44, status=training]

[EC | AAC | SVR] Best Val MSE: 0.321920 | Best Params: {'C': 1, 'epsilon': 0.1, 'gamma': 0.1}


EC | AAC | KNN:   7%|█▋                       | 330/4920 [10:49<12:52,  5.94fit/s, combo=1/16, seed=42, status=training]

[EC | AAC | DT] Best Val MSE: 0.400005 | Best Params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10}


... (86 outputs left out)

SA | QSOrder | KNN:  98%|█▉| 4844/4920 [2:51:40<00:08,  8.99fit/s, MSE=0.3373, best=-, combo=1/16, mean=0.3179, seed=45]

[SA | QSOrder | DT] Best Val MSE: 0.372362 | Best Params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10}


SA | QSOrder | KNN: 100%|█████████████████████████████████| 4920/4920 [2:51:43<00:00,  2.09s/fit, Best_Val_MSE=0.291297]

[SA | QSOrder | KNN] Best Val MSE: 0.291297 | Best Params: {'n_neighbors': 7, 'p': 1, 'weights': 'distance'}

[OK] All ML hyperparameter optimization completed.


In [5]:
# Validation Results & Best Model Selection

In [6]:
import pandas as pd
from configs.common import RESULT_DIR
from configs.ml import ML_MODELS, FEATURES

# Define paths
output_dir = RESULT_DIR / "traditional_ml"
validation_records = []

# Collect optimized validation results
for target in ["EC", "SA"]:
    for model in ML_MODELS:
        for feature in FEATURES:

            summary = pd.read_csv(
                output_dir / target
                / f"{model}_{feature}_validation_summary.csv"
            )

            # Select best hyperparameters for each feature
            best = summary.sort_values(
                "Mean_Validation_MSE"
            ).iloc[0]

            validation_records.append({
                "Dataset": target,
                "Model": model,
                "Feature": feature,
                "Best_Parameters": best["Parameters"],
                "Validation_MSE": best["Mean_Validation_MSE"],
                "Validation_SD": best["SD_Validation_MSE"]
            })

# Save all validation results
validation_results = pd.DataFrame(validation_records)

validation_results.to_csv(
    output_dir / "validation_results.csv",
    index=False
)

# Select best feature and hyperparameters per model
best_models = (
    validation_results
    .sort_values("Validation_MSE")
    .groupby(["Dataset", "Model"], as_index=False)
    .first()
    .sort_values(["Dataset", "Model"])
    .reset_index(drop=True)
)

# Save best model selection
best_models.to_csv(
    output_dir / "best_model_selection.csv",
    index=False
)

print("[OK] Validation results saved.")
print("[OK] Best features and hyperparameters selected.")

display(best_models)

[OK] Validation results saved.
[OK] Best features and hyperparameters selected.


,Dataset,Model,Feature,Best_Parameters,Validation_MSE,Validation_SD
0,EC,DT,QSOrder,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""min_...",0.391708,0.026671
1,EC,KNN,QSOrder,"{""n_neighbors"": 7, ""p"": 1, ""weights"": ""distance""}",0.297539,0.029785
2,EC,RF,QSOrder,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.279139,0.022401
3,EC,SVR,QSOrder,"{""C"": 1, ""epsilon"": 0.1, ""gamma"": 0.1}",0.280846,0.027004
4,SA,DT,QSOrder,"{""max_depth"": 10, ""min_samples_leaf"": 2, ""min_...",0.372362,0.015617
5,SA,KNN,QSOrder,"{""n_neighbors"": 7, ""p"": 1, ""weights"": ""distance""}",0.291297,0.017450
6,SA,RF,QSOrder,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.273424,0.012568
7,SA,SVR,QSOrder,"{""C"": 1, ""epsilon"": 0.1, ""gamma"": 0.1}",0.279360,0.016290


In [7]:
# Final ML Training & Test Evaluation

In [8]:

import json
import joblib
import pandas as pd
from tqdm import tqdm

from configs.common import DATA_DIR, MODEL_DIR, RESULT_DIR, RANDOM_STATE
from src.seed import set_seed
from src.evaluate import evaluate_regression

# Define paths
output_dir = RESULT_DIR / "traditional_ml"
model_dir = MODEL_DIR / "traditional_ml"

output_dir.mkdir(parents=True, exist_ok=True)

# Load selected models
best_models = pd.read_csv(
    output_dir / "best_model_selection.csv"
)

all_metrics = []
all_predictions = []

# Train and evaluate final models
with tqdm(
    best_models.itertuples(index=False),
    total=len(best_models),
    desc="Final ML Training",
    unit="model",
    ncols=120
) as pbar:

    for row in pbar:

        target = row.Dataset
        feature = row.Feature
        model_name = row.Model
        params = json.loads(row.Best_Parameters)

        # Load selected feature data
        X_train, y_train, X_test, y_test = ml.load_ml_data(
            target, feature
        )

        # Train final model
        set_seed(RANDOM_STATE)

        model = ml.create_model(
            model_name,
            params,
            seed=RANDOM_STATE
        )

        model.fit(X_train, y_train)

        # Evaluate independent test data
        predictions = model.predict(X_test)
        metrics = evaluate_regression(y_test, predictions)

        # Record test metrics
        all_metrics.append({
            "Dataset": target,
            "Model": model_name,
            "Feature": feature,
            "Best_Parameters": json.dumps(params),
            "Seed": RANDOM_STATE,
            **metrics
        })

        # Record test predictions
        sequences = pd.read_csv(
            DATA_DIR / "features" / target / f"{feature}_Test.csv"
        )["sequence"]

        pred_df = pd.DataFrame({
            "Dataset": target,
            "Model": model_name,
            "Feature": feature,
            "sequence": sequences.to_numpy(),
            "Actual_pMIC": y_test.to_numpy(),
            "Predicted_pMIC": predictions
        })

        all_predictions.append(pred_df)

        # Save trained model
        target_model_dir = model_dir / target
        target_model_dir.mkdir(parents=True, exist_ok=True)

        joblib.dump(
            model,
            target_model_dir / f"{model_name}_{feature}.joblib"
        )

        # Save test results
        pd.DataFrame(all_metrics).to_csv(
            output_dir / "test_results.csv",
            index=False
        )

        pd.concat(
            all_predictions,
            ignore_index=True
        ).to_csv(
            output_dir / "test_predictions.csv",
            index=False
        )

        # Display progress
        pbar.set_postfix(
            Dataset=target,
            Model=model_name,
            Feature=feature,
            MSE=f"{metrics['MSE']:.4f}"
        )

print("\n[OK] Final ML Training and Test Evaluation completed.")

test_results = pd.DataFrame(all_metrics)
display(test_results)


Final ML Training: 100%|███████████| 8/8 [00:15<00:00,  1.89s/model, Dataset=SA, Feature=QSOrder, MSE=0.2893, Model=SVR]


[OK] Final ML Training and Test Evaluation completed.


,Dataset,Model,Feature,Best_Parameters,Seed,MSE,RMSE,MAE,R2,Pearson_r,Pearson_p,Kendall_tau,Kendall_p
0,EC,DT,QSOrder,"{""max_depth"": 10, ""min_samples_leaf"": 1, ""min_...",42,0.378655,0.615350,0.468467,0.223603,0.500439,1.243858e-54,0.313337,4.324895e-41
1,EC,KNN,QSOrder,"{""n_neighbors"": 7, ""p"": 1, ""weights"": ""distance""}",42,0.302125,0.549659,0.422400,0.380521,0.623880,4.036616e-92,0.416181,1.414657e-72
2,EC,RF,QSOrder,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",42,0.288397,0.537026,0.407826,0.408670,0.641814,4.762355e-99,0.440686,3.802798e-81
3,EC,SVR,QSOrder,"{""C"": 1, ""epsilon"": 0.1, ""gamma"": 0.1}",42,0.304629,0.551932,0.414321,0.375387,0.613134,3.513118e-88,0.433747,1.139394e-78
4,SA,DT,QSOrder,"{""max_depth"": 10, ""min_samples_leaf"": 2, ""min_...",42,0.404849,0.636278,0.494862,0.027325,0.325568,1.469495e-19,0.199163,5.608672e-15
5,SA,KNN,QSOrder,"{""n_neighbors"": 7, ""p"": 1, ""weights"": ""distance""}",42,0.300445,0.548128,0.414138,0.278164,0.544996,6.161012e-58,0.354362,2.013453e-46
6,SA,RF,QSOrder,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",42,0.265207,0.514983,0.402615,0.362824,0.603910,4.728155e-74,0.404591,5.697214e-60
7,SA,SVR,QSOrder,"{""C"": 1, ""epsilon"": 0.1, ""gamma"": 0.1}",42,0.289296,0.537863,0.403225,0.304948,0.554687,2.276440e-60,0.383630,4.225791e-54
